# DATA PREPARATION SECTION

## Load Dataset

### Read All Subject Files

In [18]:
library(data.table)

dir <- "pamap2 physical activity monitoring/PAMAP2_Dataset/Protocol/"

paths <- list.files(path = dir,
                    pattern = "subject10[1-9]\\.dat",
                    full.names = TRUE)

dt_list <- lapply(paths, function(path) {
  dt <- fread(path, sep = " ", header = FALSE)
  subject_num <- as.integer(gsub("[^0-9]", "", basename(path)))
  dt[, subject_id := subject_num]
  return(dt)
})

### Multi-Subject Merging

In [19]:
raw_dat <- rbindlist(dt_list, use.names = FALSE, fill = TRUE)

# urutan kolom subject_id di paling depan
setcolorder(raw_dat, "subject_id")

cat("Dimensi data mentah:", dim(raw_dat)[1], "baris dan", dim(raw_dat)[2], "kolom.\n")

Dimensi data mentah: 2872533 baris dan 55 kolom.


## Data Cleaning

### Set Column Names

In [20]:
# base_cols: 4 kolom pertama
base_cols <- c("subject_id", "timestamp", "activity_id", "heart_rate")

# 17 kolom sensor untuk IMU Hand
hand_cols <- c(
  "hand_temp",
  "hand_acc_16g_x", "hand_acc_16g_y", "hand_acc_16g_z",
  "hand_acc_6g_x",  "hand_acc_6g_y",  "hand_acc_6g_z",
  "hand_gyro_x",    "hand_gyro_y",    "hand_gyro_z",
  "hand_mag_x",     "hand_mag_y",     "hand_mag_z",
  "hand_quat_1",    "hand_quat_2",    "hand_quat_3", "hand_quat_4"
)

# 17 kolom sensor untuk IMU Chest
chest_cols <- c(
  "chest_temp",
  "chest_acc_16g_x", "chest_acc_16g_y", "chest_acc_16g_z",
  "chest_acc_6g_x",  "chest_acc_6g_y",  "chest_acc_6g_z",
  "chest_gyro_x",    "chest_gyro_y",    "chest_gyro_z",
  "chest_mag_x",     "chest_mag_y",     "chest_mag_z",
  "chest_quat_1",    "chest_quat_2",    "chest_quat_3", "chest_quat_4"
)

# 17 kolom sensor untuk IMU Ankle
ankle_cols <- c(
  "ankle_temp",
  "ankle_acc_16g_x", "ankle_acc_16g_y", "ankle_acc_16g_z",
  "ankle_acc_6g_x",  "ankle_acc_6g_y",  "ankle_acc_6g_z",
  "ankle_gyro_x",    "ankle_gyro_y",    "ankle_gyro_z",
  "ankle_mag_x",     "ankle_mag_y",     "ankle_mag_z",
  "ankle_quat_1",    "ankle_quat_2",    "ankle_quat_3", "ankle_quat_4"
)

official_colnames <- c(base_cols, hand_cols, chest_cols, ankle_cols)
colnames(raw_dat) <- official_colnames

### Remove Activity 0

In [21]:
row_count <- nrow(raw_dat) # before cleaning activity 0
clean_dat <- raw_dat[activity_id != 0]
clean_row_count <- nrow(clean_dat) # after cleaning activity 0

rows_removed <- row_count - clean_row_count
pct_removed <- (rows_removed / row_count) * 100

cat("Jumlah baris awal mentah    :", row_count, "\n")
cat("Jumlah baris setelah filter :", clean_row_count, "\n")
cat("Total baris yang dibuang    :", rows_removed, "\n")
cat("Persentase data dibuang     :", round(pct_removed, 2), "%\n")

Jumlah baris awal mentah    : 2872533 
Jumlah baris setelah filter : 1942872 
Total baris yang dibuang    : 929661 
Persentase data dibuang     : 32.36 %


### Handle Missing Value

In [22]:
# daftar kolom orientasi dan timestamp yang akan didrop
cols_to_drop <- c(
  "timestamp",
  "hand_quat_1", "hand_quat_2", "hand_quat_3", "hand_quat_4",
  "chest_quat_1", "chest_quat_2", "chest_quat_3", "chest_quat_4",
  "ankle_quat_1", "ankle_quat_2", "ankle_quat_3", "ankle_quat_4"
)

# hapus kolom langsung
clean_dat[, (cols_to_drop) := NULL]

# hitung jumlah Missing Value (NA)
na_before <- sum(is.na(clean_dat))
cat("Total nilai NA sebelum imputasi:", na_before, "\n")

# Imputasi Median
# cari kolom mana saja yang mendeteksi adanya nilai NA (terutama heart_rate)
columns_with_na <- names(clean_dat)[sapply(clean_dat, anyNA)]

# looping pengisian nilai NA dengan median menggunakan fungsi set() yang super hemat RAM
for (col in columns_with_na) {
  # Hitung nilai median dari kolom tersebut (abaikan nilai NA saat menghitung)
  median_value <- median(clean_dat[[col]], na.rm = TRUE)
  # Cari baris yang NA pada kolom tersebut, lalu timpa dengan nilai median
  set(clean_dat, i = which(is.na(clean_dat[[col]])), j = col, value = median_value)
}

# verifikasi kembali jumlah NA setelah imputasi
na_after <- sum(is.na(clean_dat))

cat("Kolom yang diimputasi         :", paste(columns_with_na, collapse = ", "), "\n")
cat("Total nilai NA setelah proses :", na_after, "\n")
cat("Status data                   :", if(na_after == 0) "BERSIH (0 NA)" else "MASIH ADA NA", "\n")

Total nilai NA sebelum imputasi: 2052127 
Kolom yang diimputasi         : heart_rate, hand_temp, hand_acc_16g_x, hand_acc_16g_y, hand_acc_16g_z, hand_acc_6g_x, hand_acc_6g_y, hand_acc_6g_z, hand_gyro_x, hand_gyro_y, hand_gyro_z, hand_mag_x, hand_mag_y, hand_mag_z, chest_temp, chest_acc_16g_x, chest_acc_16g_y, chest_acc_16g_z, chest_acc_6g_x, chest_acc_6g_y, chest_acc_6g_z, chest_gyro_x, chest_gyro_y, chest_gyro_z, chest_mag_x, chest_mag_y, chest_mag_z, ankle_temp, ankle_acc_16g_x, ankle_acc_16g_y, ankle_acc_16g_z, ankle_acc_6g_x, ankle_acc_6g_y, ankle_acc_6g_z, ankle_gyro_x, ankle_gyro_y, ankle_gyro_z, ankle_mag_x, ankle_mag_y, ankle_mag_z 
Total nilai NA setelah proses : 0 
Status data                   : BERSIH (0 NA) 


### Activity ID Mapping

In [23]:
# buat vektor representasi lengkap kode ID dan nama aktivitasnya
activity_map <- c(
  "1"  = "Lying",
  "2"  = "Sitting",
  "3"  = "Standing",
  "4"  = "Walking",
  "5"  = "Running",
  "6"  = "Cycling",
  "7"  = "Nordic Walking",
  "12" = "Ascending Stairs",
  "13" = "Descending Stairs",
  "16" = "Vacuum Cleaning",
  "17" = "Ironing",
  "24" = "Rope Jumping"
)

# tambah kolom baru 'activity_name' pakai operasi data.table
clean_dat[, activity_name := activity_map[as.character(activity_id)]]

# ubah kolom nama aktivitas menjadi tipe Factor
clean_dat[, activity_name := as.factor(activity_name)]

# hapus kolom asli 'activity_id' karena sudah digantikan oleh 'activity_name'
clean_dat[, activity_id := NULL]

cat("Tipe data kolom 'activity_name':", class(clean_dat$activity_name), "\n\n")
cat("Daftar aktivitas unik yang terdeteksi:\n")
print(levels(clean_dat$activity_name))

Tipe data kolom 'activity_name': factor 

Daftar aktivitas unik yang terdeteksi:
 [1] "Ascending Stairs"  "Cycling"           "Descending Stairs"
 [4] "Ironing"           "Lying"             "Nordic Walking"   
 [7] "Rope Jumping"      "Running"           "Sitting"          
[10] "Standing"          "Vacuum Cleaning"   "Walking"          


### Select Features for Clustering

In [24]:
# tentukan nama kolom yang bertindak sebagai metadata/label (y)
metadata_cols <- c("subject_id", "activity_name")

# buat Objek y (Metadata & Label)
pamap2_y <- clean_dat[, ..metadata_cols]

# buat Objek X (Fitur Sensor Numerik)
pamap2_X <- clean_dat[, !..metadata_cols]

# Log Informasi Pemisahan
cat("Dimensi Matriks Fitur (X) :", dim(pamap2_X)[1], "baris dan", dim(pamap2_X)[2], "kolom\n")
cat("Dimensi Matriks Label (y) :", dim(pamap2_y)[1], "baris dan", dim(pamap2_y)[2], "kolom\n\n")

cat("Daftar 5 Fitur Pertama di X:\n")
print(names(pamap2_X)[1:5])

cat("\nDaftar Fitur di y:\n")
print(names(pamap2_y))

Dimensi Matriks Fitur (X) : 1942872 baris dan 40 kolom
Dimensi Matriks Label (y) : 1942872 baris dan 2 kolom

Daftar 5 Fitur Pertama di X:
[1] "heart_rate"     "hand_temp"      "hand_acc_16g_x" "hand_acc_16g_y"
[5] "hand_acc_16g_z"

Daftar Fitur di y:
[1] "subject_id"    "activity_name"


## Data Summary & Verification

### Structure & Dimension Check

In [25]:
rows_X <- nrow(pamap2_X)
rows_y <- nrow(pamap2_y)
cols_X <- ncol(pamap2_X)
cols_y <- ncol(pamap2_y)

cat("====== VERIFIKASI DIMENSI MATRIKS ======\n")
cat("Matriks Fitur (X)      :", rows_X, "baris dan", cols_X, "kolom.\n")
cat("Matriks Label/Target (y):", rows_y, "baris dan", cols_y, "kolom.\n")
cat("Status Sinkronisasi Baris:", if(rows_X == rows_y) "SINKRON (Aman)" else "ERROR: Jumlah baris tidak sama!", "\n")
cat("========================================\n\n")

cat("====== STRUKTUR TIPE DATA MATRIKS X ======\n")
print(str(pamap2_X, list.len = 15)) # list.len agar output tidak terlalu panjang
cat("==========================================\n\n")

cat("====== STRUKTUR TIPE DATA MATRIKS y ======\n")
print(str(pamap2_y))
cat("==========================================\n")

====== VERIFIKASI DIMENSI MATRIKS ======
Matriks Fitur (X)      : 1942872 baris dan 40 kolom.
Matriks Label/Target (y): 1942872 baris dan 2 kolom.
Status Sinkronisasi Baris: SINKRON (Aman) 

====== STRUKTUR TIPE DATA MATRIKS X ======
Classes 'data.table' and 'data.frame':	1942872 obs. of  40 variables:
 $ heart_rate     : num  104 104 104 104 100 104 104 104 104 104 ...
 $ hand_temp      : num  30.4 30.4 30.4 30.4 30.4 ...
 $ hand_acc_16g_x : num  2.22 2.29 2.29 2.22 2.3 ...
 $ hand_acc_16g_y : num  8.28 7.67 7.14 7.14 7.26 ...
 $ hand_acc_16g_z : num  5.59 5.74 5.82 5.9 6.09 ...
 $ hand_acc_6g_x  : num  2.25 2.27 2.27 2.22 2.21 ...
 $ hand_acc_6g_y  : num  8.55 8.15 7.66 7.26 7.24 ...
 $ hand_acc_6g_z  : num  5.77 5.79 5.79 5.88 5.96 ...
 $ hand_gyro_x    : num  -0.00475 -0.17171 -0.23824 -0.19291 -0.06996 ...
 $ hand_gyro_y    : num  0.0376 0.0255 0.0112 0.0191 -0.0183 ...
 $ hand_gyro_z    : num  -0.011145 -0.009538 0.000831 0.013374 0.004582 ...
 $ hand_mag_x     : num  8.93 9.58 9

### Missing Value Final Check

In [28]:
# Cek Sisa Missing Value (NA)
total_na_X <- sum(is.na(pamap2_X))
total_na_y <- sum(is.na(pamap2_y))

# Cek Infinite Values (Khusus untuk Matriks Fitur X yang bertipe numerik)
total_inf_X <- sum(sapply(pamap2_X, function(col) sum(is.infinite(col))))

# Cek Baris Duplikat
has_duplicate <- anyDuplicated(pamap2_X)
total_duplicates <- if(has_duplicate > 0) sum(duplicated(pamap2_X)) else 0

# Menampilkan Ringkasan Log Integritas Data
cat("====== LENGKAP LOG INTEGRITAS DATA FINAL ======\n")
cat("Sisa Nilai NA di Matriks X     :", total_na_X, "\n")
cat("Sisa Nilai NA di Matriks y     :", total_na_y, "\n")
cat("Total Nilai Infinite di X      :", total_inf_X, "\n")
cat("Total Baris Duplikat di X      :", total_duplicates, "\n")
cat("-----------------------------------------------\n")
cat("Kesimpulan Kelayakan Data      :", 
    if(total_na_X == 0 && total_na_y == 0 && total_inf_X == 0) {
      "SIAP PAKAI (Data Bersih Berstandar Tinggi)"
    } else {
      "PERLU DIPERIKSA ULANG (Masih ada anomali)"
    }, "\n")
cat("===============================================\n")

# Catatan: Jika ditemukan baris duplikat pada data sensor fisik yang terekam kontinu (100 Hz), 
# hal tersebut wajar terjadi apabila subjek berada dalam posisi diam sempurna (misal: Lying atau Sitting). 
# Oleh karena itu, baris duplikat pada sensor tidak wajib didrop kecuali jika mengganggu performa.

====== LENGKAP LOG INTEGRITAS DATA FINAL ======
Sisa Nilai NA di Matriks X     : 0 
Sisa Nilai NA di Matriks y     : 0 
Total Nilai Infinite di X      : 0 
Total Baris Duplikat di X      : 124 
-----------------------------------------------
Kesimpulan Kelayakan Data      : SIAP PAKAI (Data Bersih Berstandar Tinggi) 


## Save Clean Data

In [29]:
# Menyimpan Matriks Fitur X
fwrite(pamap2_X, "X_prepared.csv", row.names = FALSE)
cat("-> Berhasil menyimpan: X_prepared.csv\n")

# Menyimpan Matriks Label y
fwrite(pamap2_y, "y_labels.csv", row.names = FALSE)
cat("-> Berhasil menyimpan: y_labels.csv\n\n")

cat("====== PROSES DATA PREPARATION SELESAI ======\n")
cat("Berkas 'X_prepared.csv' dan 'y_labels.csv' siap digunakan pada tahap selanjutnya.\n")
cat("=============================================\n")

-> Berhasil menyimpan: X_prepared.csv
-> Berhasil menyimpan: y_labels.csv

====== PROSES DATA PREPARATION SELESAI ======
Berkas 'X_prepared.csv' dan 'y_labels.csv' siap digunakan pada tahap selanjutnya.
